# ETL — Minnie65 CSM Cell-Type Taxonomy & Cluster Membership

Registers the Minnie65 column-soma-morphology (CSM) cell-type taxonomy as a global
hierarchy and writes per-cell `ClusterMembership` for `dataset_id="minnie65_v1412_csm_cluster"`.

Outputs (under `../scratch/em_patchseq_wnm_v1/`):
- `algorithmrun/` — one row, `id="minnie65_csm_clustering"`.
- `clusterhierarchy/` — one row, `id="minnie65_csm_cell_types"`.
- `cluster/` — 1 root + 2 classes + 13 leaf cell-types = 16 rows, `hierarchy_id="minnie65_csm_cell_types"`.
- `clustermembership/` — one row per (cell × ancestor) under predicate
  `project_id='minnie65' AND hierarchy_id='minnie65_csm_cell_types'`.

## Caveats and provenance — please read

The intended source per `etl_minnie_02_cell_features.ipynb`'s narrative is
`../data/minnie1412/minnie_features.parquet` (`cell_type` column). The reference
notebook `code/parse_minnie_clustering.ipynb` instead pulls the taxonomy from the
CAVE table `cell_type_multifeature_v1`. We considered both and rejected both:

1. **CAVE table is gone at v1412.** Direct check returns
   `404 NOT FOUND ... No table named 'cell_type_multifeature_v1' for version '1412'`
   in datastack `minnie65_phase3_v1`. None of the cell-type tables present at
   v1412 obviously share the reference's vocabulary.
2. **Parquet vocabulary differs from the reference palette.** The parquet has
   `L2a/L2b/L2c/L3a/L3b/L4a/L4b/L4c/L5a/L5b/L5ET/L5NP/L6tall-{a,b,c}/L6short-{a,b}`,
   while the reference's hardcoded palette uses
   `L2IT/L3IT/L4IT/L5IT/L5ET/L5NP/L6CT/L6IT/L6SP/DTC/ITC/PTC/STC`. They share only
   `L5ET` and `L5NP`. Mapping one to the other would require fabrication.

**Temp solution (per user direction):** read the already-built legacy delta
lakes at `../data/microns1412/cluster/` and `../data/microns1412/clustermembership/`,
which encode exactly the reference notebook's outputs (16-row hierarchy,
parent-propagated memberships, hex colors), and translate them to the current
schema. Specifically:

- Drop the legacy `Cluster.project_id` slot (Cluster is now global; not
  ProjectScoped).
- Drop the legacy typo column `heirachy_category` (current schema uses
  `hierarchy_category`; we leave it null since the legacy values
  `major_class/class/subtype` don't match the slot's intent and we don't have a
  registered HierarchyCategory vocabulary yet — see TODO).
- Add `hierarchy_id="minnie65_csm_cell_types"` to every Cluster row.
- For ClusterMembership, add `hierarchy_id="minnie65_csm_cell_types"` and keep
  `item`, `cluster`, `project_id`, `probability` as-is.

This faithfully reproduces the reference notebook's clustering output without
re-running any algorithm here.

## TODO — `HierarchyCategory`

`HierarchyCategory` is global with no `project_id`/`hierarchy_id` discriminator,
and category ids like `class`/`subclass`/`cluster` are intentionally shared
across taxonomies. A scoped overwrite from this notebook would clobber Tasic's
rows; an append would collide on id. Skipping for now — needs a global-dedup
append helper before any taxonomy notebook should write here.

## Idempotency

All writes use predicates that scope to either the hierarchy (for global
tables) or `(project_id, hierarchy_id)` (for `clustermembership/`).
Re-running this notebook overwrites only this taxonomy's rows.


In [1]:
import pandas as pd
import polars as pl

from connects_common_connectivity.models import (
    AlgorithmRun,
    Cluster,
    ClusterHierarchy,
    ClusterMembership,
)
from connects_common_connectivity.config import output_root
from connects_common_connectivity.io import write_models


In [2]:
OUTPUT_ROOT  = output_root()
PROJECT_ID   = "minnie65"
DATASET_ID   = "minnie65_v1300_csm_cluster"
HIERARCHY_ID = "minnie65_csm_cell_types"
RUN_ID       = "minnie65_csm_clustering"
ROOT_ID      = "neuron"

LEGACY_CLUSTER_PATH    = "../data/microns1412/cluster"
LEGACY_MEMBERSHIP_PATH = "../data/microns1412/clustermembership"

print(f"OUTPUT_ROOT  : {OUTPUT_ROOT}")
print(f"PROJECT_ID   : {PROJECT_ID}")
print(f"DATASET_ID   : {DATASET_ID}")
print(f"HIERARCHY_ID : {HIERARCHY_ID}")
print(f"RUN_ID       : {RUN_ID}")


OUTPUT_ROOT  : ../scratch/em_patchseq_wnm_v2/
PROJECT_ID   : minnie65
DATASET_ID   : minnie65_v1300_csm_cluster
HIERARCHY_ID : minnie65_csm_cell_types
RUN_ID       : minnie65_csm_clustering


## Read legacy delta lakes (`data/microns1412/`)

In [3]:
legacy_clu = pl.read_delta(LEGACY_CLUSTER_PATH)
legacy_mem = pl.read_delta(LEGACY_MEMBERSHIP_PATH)

print("legacy cluster:", legacy_clu.shape, legacy_clu.columns)
print("legacy membership:", legacy_mem.shape, legacy_mem.columns)
print()
print("legacy clusters by level:")
print(legacy_clu.group_by("level").len().sort("level"))


legacy cluster: (16, 9) ['id', 'parent', 'children', 'level', 'score', 'hex_color', 'heirachy_category', 'distance_to_parent', 'project_id']
legacy membership: (107340, 6) ['item', 'cluster', 'membership_score', 'probability', 'distance', 'project_id']

legacy clusters by level:
shape: (3, 2)
┌───────┬─────┐
│ level ┆ len │
│ ---   ┆ --- │
│ i64   ┆ u32 │
╞═══════╪═════╡
│ 0     ┆ 1   │
│ 1     ┆ 2   │
│ 2     ┆ 13  │
└───────┴─────┘


## Prerequisite check — every membership cell must be a registered DataItem in the CSM cohort

In [4]:
assoc = (
    pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID))
)
assert assoc.shape[0] > 0, (
    f"No DataItemDataSetAssociation rows for ({PROJECT_ID}, {DATASET_ID}); run _02 first."
)
csm_cell_ids = set(assoc["dataitem_id"].to_list())
print(f"Registered CSM cells: {len(csm_cell_ids):,}")

membership_items = set(legacy_mem["item"].unique().to_list())
missing = membership_items - csm_cell_ids
assert not missing, (
    f"{len(missing)} legacy membership ids are not registered DataItems for the CSM cohort. "
    f"Examples: {sorted(missing)[:5]}"
)
print(f"All {len(membership_items):,} unique membership cells are registered DataItems.")


Registered CSM cells: 35,783
All 35,780 unique membership cells are registered DataItems.


## Build `Cluster` rows from the legacy hierarchy

Drop the legacy `project_id` and `heirachy_category` columns (former: schema
change made `Cluster` global; latter: legacy typo, and the legacy values
`major_class/class/subtype` are not the intended HierarchyCategory ids — leave
`hierarchy_category` null until the global-dedup write path is in place).
Stamp `hierarchy_id` on every row.

In [5]:
legacy_clu_pd = legacy_clu.to_pandas()

cluster_rows: list[Cluster] = []
for _, row in legacy_clu_pd.iterrows():
    children = row["children"]
    if children is None or (isinstance(children, float) and pd.isna(children)):
        children_val = None
    else:
        children_val = [str(c) for c in list(children)]
    cluster_rows.append(Cluster(
        id=str(row["id"]),
        hierarchy_id=HIERARCHY_ID,
        parent=None if pd.isna(row["parent"]) or row["parent"] is None or row["parent"] == "" else str(row["parent"]),
        children=children_val,
        level=int(row["level"]),
        score=None,
        hex_color=str(row["hex_color"]),
        hierarchy_category=None,
        distance_to_parent=None,
    ))

assert len(cluster_rows) == legacy_clu.shape[0]
assert {c.id for c in cluster_rows if c.parent is None} == {ROOT_ID}, "expected single root 'neuron'"
print(f"Built {len(cluster_rows)} Cluster rows; root={ROOT_ID}")
print("level counts:", pd.Series([c.level for c in cluster_rows]).value_counts().sort_index().to_dict())


Built 16 Cluster rows; root=neuron
level counts: {0: 1, 1: 2, 2: 13}


## Write `cluster/`

In [6]:
result = write_models(cluster_rows, output_root=OUTPUT_ROOT)
print(f"Cluster written: {result.rows_written} rows")

verify_clu = (
    pl.read_delta(OUTPUT_ROOT + "cluster/")
      .filter(pl.col("hierarchy_id") == HIERARCHY_ID)
)
print("verify shape:", verify_clu.shape)
assert verify_clu.shape[0] == len(cluster_rows)
assert set(verify_clu["id"].to_list()) == {c.id for c in cluster_rows}


Cluster written: 16 rows
verify shape: (16, 9)


## `AlgorithmRun`

In [7]:
run_row = AlgorithmRun(
    id=RUN_ID,
    algorithm_name="CSM (column-soma-morphology) cell-type clustering, Minnie65 v1300",
    algorithm_version="v1300",
    score_description=None,
    distance_description=None,
    run_timestamp=None,
    json_object=None,
    input_dataset=DATASET_ID,
    # produced_hierarchies omitted: inlined dict per schema; ClusterHierarchy.run carries the link.
)

result = write_models([run_row], output_root=OUTPUT_ROOT)
print(f"algorithmrun/ written: {result.rows_written} rows")
verify_r = pl.read_delta(OUTPUT_ROOT + "algorithmrun/").filter(pl.col("id") == RUN_ID)
assert verify_r.shape[0] == 1


algorithmrun/ written: 1 rows


## `ClusterHierarchy`

In [8]:
hierarchy_row = ClusterHierarchy(
    id=HIERARCHY_ID,
    run=RUN_ID,
    root=ROOT_ID,
    clusters=[c.id for c in cluster_rows],
)
result = write_models([hierarchy_row], output_root=OUTPUT_ROOT)
print(f"ClusterHierarchy written: {result.rows_written} rows")
verify_h = pl.read_delta(OUTPUT_ROOT + "clusterhierarchy/").filter(pl.col("id") == HIERARCHY_ID)
assert verify_h.shape[0] == 1


ClusterHierarchy written: 1 rows


## `ClusterMembership` — translate legacy rows to current schema

The legacy delta lake already encodes one row per (cell × ancestor) — the
parent propagation is baked in. We just stamp `hierarchy_id` and validate that
every `cluster` value is a known cluster id.

In [9]:
valid_cluster_ids = {c.id for c in cluster_rows}
unknown = set(legacy_mem["cluster"].unique().to_list()) - valid_cluster_ids
assert not unknown, f"legacy membership references unknown cluster ids: {sorted(unknown)}"
print(f"All {len(set(legacy_mem['cluster'].unique().to_list()))} cluster labels in legacy memberships are valid.")

legacy_mem_pd = legacy_mem.to_pandas()

memberships: list[ClusterMembership] = []
for _, row in legacy_mem_pd.iterrows():
    memberships.append(ClusterMembership(
        item=str(row["item"]),
        cluster=str(row["cluster"]),
        membership_score=None if pd.isna(row.get("membership_score")) else float(row["membership_score"]),
        probability=None if pd.isna(row.get("probability")) else float(row["probability"]),
        distance=None if pd.isna(row.get("distance")) else float(row["distance"]),
        project_id=PROJECT_ID,
        hierarchy_id=HIERARCHY_ID,
    ))

print(f"Built {len(memberships):,} ClusterMembership rows")


All 16 cluster labels in legacy memberships are valid.
Built 107,340 ClusterMembership rows


## Write `clustermembership/`

In [10]:
result = write_models(memberships, output_root=OUTPUT_ROOT)
print(f"ClusterMembership written: {result.rows_written} rows")

verify_cm = (
    pl.read_delta(OUTPUT_ROOT + "clustermembership/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("hierarchy_id") == HIERARCHY_ID))
)
print("verify shape:", verify_cm.shape)
assert verify_cm.shape[0] == len(memberships)
assert set(verify_cm["cluster"].unique().to_list()) <= valid_cluster_ids
print(f"unique cells: {verify_cm['item'].n_unique():,}")
print("rows per cluster:")
print(verify_cm.group_by("cluster").len().sort("len", descending=True))


ClusterMembership written: 107340 rows
verify shape: (107340, 7)
unique cells: 35,780
rows per cluster:
shape: (16, 2)
┌───────────────┬───────┐
│ cluster       ┆ len   │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ neuron        ┆ 35780 │
│ glutamatergic ┆ 31605 │
│ L4IT          ┆ 7348  │
│ L6CT          ┆ 7245  │
│ L2IT          ┆ 5038  │
│ …             ┆ …     │
│ L5ET          ┆ 865   │
│ ITC           ┆ 662   │
│ STC           ┆ 460   │
│ L5NP          ┆ 267   │
│ L6SP          ┆ 176   │
└───────────────┴───────┘


## Summary

| Path | Rows |
|---|---|
| `algorithmrun/` | +1 (`minnie65_csm_clustering`) |
| `clusterhierarchy/` | +1 (`minnie65_csm_cell_types`) |
| `cluster/` (`hierarchy_id='minnie65_csm_cell_types'`) | 16 (1 root + 2 classes + 13 leaves) |
| `clustermembership/` (`project_id='minnie65', hierarchy_id='minnie65_csm_cell_types'`) | one per (cell × ancestor) |

**Source:** legacy delta lakes at `data/microns1412/cluster/` and
`data/microns1412/clustermembership/`, translated to the current schema. See
header for why neither CAVE nor `minnie_features.parquet` is used as the
clustering source here.
